### Application of FUSE Algorithm to Cora Dataset

#### Dataset Split: 30-70 MCAR


## Results Summary

|  Classifier | Avg Accuracy |  Avg F1 |
|---|---:|---:|
| MLP | **0.7780** | **0.7797** |
| Multinomial Logistic Regression | **0.7780** | **0.7790** |





### Emebdding generation (algorithm iterations run time) : ~ 0.70 s

In [ ]:
!pip install ogb
!pip install torch_geometric



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 2.6 MB/s eta 0:00:00
  Using cached torch_geometric-2.8.0-py3-none-any.whl.metadata (64 kB)
Using cached torch_geometric-2.8.0-py3-none-any.whl (1.3 MB)


In [ ]:

import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch_geometric.datasets import Planetoid
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

seed = 123
torch.manual_seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

Using device: cuda


## Loading dataset:

In [ ]:
dataset = Planetoid(root="Cora", name="Cora")

data = dataset[0]

print(data)
print("Dataset classes:", dataset.num_classes)
print("Feature dimension:", dataset.num_node_features)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Dataset classes: 7
Feature dimension: 1433


In [ ]:


edge_index = data.edge_index.long()
node_features = data.x.float()
labels_tensor = data.y.long().view(-1)

num_nodes = data.num_nodes
num_classes = dataset.num_classes


split_ratio = 0.70


test_fraction = split_ratio

all_indices = np.arange(num_nodes)
all_labels = labels_tensor.cpu().numpy()


train_np, test_np = train_test_split(
    all_indices,
    test_size=test_fraction,
    random_state= seed,
    shuffle=True,
    stratify=all_labels,
)

train_idx = torch.tensor(np.sort(train_np), dtype=torch.long)
test_idx = torch.tensor(np.sort(test_np), dtype=torch.long)

valid_idx = None

print(f"Nodes: {num_nodes}")
print(f"Edges in edge_index: {edge_index.shape[1]}")
print(f"Classes: {num_classes}")
print(f"Node feature matrix: {node_features.shape}")
print(f"Labels tensor: {labels_tensor.shape}")
print()
print(f"Train / labelled nodes: {len(train_idx)}")
print(f"Test / masked nodes:    {len(test_idx)}")
print(f"Train fraction: {len(train_idx) / num_nodes:.3f}")
print(f"Test fraction:  {len(test_idx) / num_nodes:.3f}")

print("\nClass counts in full data:")
print(torch.bincount(labels_tensor, minlength=num_classes).tolist())

print("\nClass counts in train split:")
print(torch.bincount(labels_tensor[train_idx], minlength=num_classes).tolist())

print("\nClass counts in test split:")
print(torch.bincount(labels_tensor[test_idx], minlength=num_classes).tolist())

Nodes: 2708
Edges in edge_index: 10556
Classes: 7
Node feature matrix: torch.Size([2708, 1433])
Labels tensor: torch.Size([2708])

Train / labelled nodes: 812
Test / masked nodes:    1896
Train fraction: 0.300
Test fraction:  0.700

Class counts in full data:
[351, 217, 418, 818, 426, 298, 180]

Class counts in train split:
[105, 65, 125, 245, 128, 90, 54]

Class counts in test split:
[246, 152, 293, 573, 298, 208, 126]


## Masking labels

In [ ]:
n = num_nodes

is_labeled = torch.zeros(n, dtype=torch.bool, device=device)
is_labeled[train_idx.to(device)] = True

is_masked = torch.zeros(n, dtype=torch.bool, device=device)
is_masked[test_idx.to(device)] = True


labels_masked = torch.full((n,), -1, dtype=torch.long, device=device)
labels_masked[train_idx.to(device)] = labels_tensor[train_idx].to(device)

print(f"Total Nodes: {n}")
print(f"Labeled nodes / train: {(labels_masked != -1).sum().item()}")
print(f"Masked nodes / test:   {(labels_masked == -1).sum().item()}")
print(f"Masked labels shape: {labels_masked.shape}")

Total Nodes: 2708
Labeled nodes / train: 812
Masked nodes / test:   1896
Masked labels shape: torch.Size([2708])


## Building sparse adjacency matrix:


In [ ]:
self_loop_mask = edge_index[0] != edge_index[1]
edge_index_clean = edge_index[:, self_loop_mask].contiguous()

m = edge_index_clean.shape[1] / 2

edge_index_device = edge_index_clean.to(device)
vals = torch.ones(edge_index_device.shape[1], dtype=torch.float32, device=device)

A_sparse = torch.sparse_coo_tensor(
    edge_index_device,
    vals,
    (n, n),
    device=device
).coalesce().to_sparse_csr()

deg = torch.zeros(n, dtype=torch.float32, device=device)
deg.scatter_add_(
    0,
    edge_index_device[0],
    torch.ones(edge_index_device.shape[1], dtype=torch.float32, device=device)
)

d = deg.unsqueeze(1)

print(f"A_sparse: {A_sparse.shape}, nnz={A_sparse._nnz()}")
print(f"Degree vector d: {d.shape}")
print(f"m used in modularity denominator: {m}")
print(f"Number of isolated nodes: {(deg == 0).sum().item()}")

/tmp/ipykernel_701/4284759995.py:9: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  A_sparse = torch.sparse_coo_tensor(


A_sparse: torch.Size([2708, 2708]), nnz=10556
Degree vector d: torch.Size([2708, 1])
m used in modularity denominator: 5278.0
Number of isolated nodes: 0


/tmp/ipykernel_701/4284759995.py:14: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  ).coalesce().to_sparse_csr()


## initializing FUSE embeddings

In [ ]:


hyper_params = { # based on those in original paper

        "k": 145,
        "lr": 0.31,
        "lambda_sup": 0.6,
        "lambda_semi": 1.9,
        "T_epochs": 200,
        "r": 20,
        "L": 4,
        "Lp": 1,

}

embedding_dim = hyper_params["k"]


# random embedding init:
S = torch.randn(n, embedding_dim, device=device)
Q, _ = torch.linalg.qr(S, mode="reduced")
S = Q.contiguous()

print("Initial embedding matrix S:", S.shape)


Initial embedding matrix S: torch.Size([2708, 145])


## Building CSR graph for batched random walks

In [ ]:
def build_csr_graph(edge_index: torch.Tensor, n: int, device):

    src, dst = edge_index[0], edge_index[1]

    order = torch.argsort(src)
    src_s = src[order]
    dst_s = dst[order]

    deg_cpu = torch.bincount(src_s.cpu(), minlength=n)

    row_ptr_cpu = torch.zeros(n + 1, dtype=torch.long)
    row_ptr_cpu[1:] = torch.cumsum(deg_cpu, dim=0)

    row_ptr = row_ptr_cpu.to(device)
    col_idx = dst_s.to(device)
    deg_vec = deg_cpu.to(device)

    return row_ptr, col_idx, deg_vec

row_ptr, col_idx, deg_vec = build_csr_graph(edge_index_clean, n, device)

print(f"row_ptr: {row_ptr.shape}")
print(f"col_idx: {col_idx.shape}")
print(f"deg_vec: {deg_vec.shape}")

row_ptr: torch.Size([2709])
col_idx: torch.Size([10556])
deg_vec: torch.Size([2708])


In [ ]:
def batched_csr_random_walk(
    row_ptr,
    col_idx,
    deg_vec,
    labels_masked,
    is_masked,
    r=20,
    L=4,
    Lp=1,
    device="cpu"
):


    unlab_ids = torch.where(is_masked)[0]
    n_unlab = unlab_ids.shape[0]

    walk_neigh = torch.full((n_unlab, r, Lp), -1, dtype=torch.long, device=device)
    labeled_count = torch.zeros(n_unlab, r, dtype=torch.long, device=device)

    current = unlab_ids.unsqueeze(1).expand(n_unlab, r).clone()

    for step in range(L):
        deg_curr = deg_vec[current]
        active = deg_curr > 0

        next_node = current.clone()

        if active.any():
            deg_active = deg_curr[active]
            offsets = (torch.rand(deg_active.shape, device=device) * deg_active.float()).long()
            flat_indices = row_ptr[current[active]] + offsets
            next_node[active] = col_idx[flat_indices]

        next_is_lab = labels_masked[next_node] != -1
        slot_open = labeled_count < Lp
        record = next_is_lab & slot_open

        lc_clamped = labeled_count.clamp(max=Lp - 1)

        old_values = walk_neigh.gather(2, lc_clamped.unsqueeze(2))
        new_values = torch.where(
            record.unsqueeze(2),
            next_node.unsqueeze(2),
            old_values
        )

        walk_neigh.scatter_(2, lc_clamped.unsqueeze(2), new_values)

        labeled_count = labeled_count + record.long()
        current = next_node

    return walk_neigh, unlab_ids


walk_neigh, unlab_ids = batched_csr_random_walk(
    row_ptr=row_ptr,
    col_idx=col_idx,
    deg_vec=deg_vec,
    labels_masked=labels_masked,
    is_masked=is_masked,
    r=hyper_params["r"],
    L=hyper_params["L"],
    Lp=hyper_params["Lp"],
    device=device
)

valid_recorded = (walk_neigh >= 0).sum().item()
valid_per_node = (walk_neigh >= 0).view(walk_neigh.shape[0], -1).sum(dim=1)

print("walk_neigh:", walk_neigh.shape)
print("unlab_ids:", unlab_ids.shape)
print("Valid recorded labelled neighbours:", valid_recorded)
print("Average labelled candidates per test node:", valid_per_node.float().mean().item())
print("Fraction of test nodes with zero labelled candidates:", (valid_per_node == 0).float().mean().item())

walk_neigh: torch.Size([1896, 20, 1])
unlab_ids: torch.Size([1896])
Valid recorded labelled neighbours: 23009
Average labelled candidates per test node: 12.135547637939453
Fraction of test nodes with zero labelled candidates: 0.04219409078359604


## 7. FUSE optimization losses

In [ ]:
def modularity_grad_prop(A_sparse, d, S, m, grad_out):


    if d.dim() == 1:
        d = d.unsqueeze(1).float()
    elif d.dtype != S.dtype:
        d = d.float()

    grad_out.copy_(torch.sparse.mm(A_sparse, S))

    global_sum = S.sum(dim=0, keepdim=True)  # 1^T S, shape [1, k]
    grad_out.add_(d @ global_sum, alpha=-(1.0 / (2.0 * m)))

    grad_out.mul_(1.0 / (2.0 * m))


def supervised_grad(S, labels_masked, is_labeled, num_classes, grad_out):

    grad_out.zero_()

    lab_idx = torch.where(is_labeled)[0]
    lab_cls = labels_masked[lab_idx]
    lab_emb = S[lab_idx]

    class_sum = torch.zeros(num_classes, S.shape[1], device=S.device)
    class_count = torch.zeros(num_classes, device=S.device)

    class_sum.scatter_add_(0, lab_cls.unsqueeze(1).expand_as(lab_emb), lab_emb)
    class_count.scatter_add_(0, lab_cls, torch.ones(lab_idx.shape[0], device=S.device))

    mu = class_sum / class_count.unsqueeze(1).clamp(min=1)
    grad_out[lab_idx] = lab_emb - mu[lab_cls]


def semi_sup_grad(S, walk_neigh, unlab_ids, grad_out, batch_size=25000):

    grad_out.zero_()

    n_unlab = unlab_ids.shape[0]
    P = walk_neigh.shape[1] * walk_neigh.shape[2]

    flat_neigh = walk_neigh.view(n_unlab, P)
    valid_mask = flat_neigh >= 0
    safe_neigh = flat_neigh.clamp(min=0)

    for i in range(0, n_unlab, batch_size):
        end = min(i + batch_size, n_unlab)

        ids_batch = unlab_ids[i:end]
        safe_neigh_batch = safe_neigh[i:end]
        valid_mask_batch = valid_mask[i:end]

        S_i = S[ids_batch]
        S_j = S[safe_neigh_batch]

        sim = torch.bmm(S_i.unsqueeze(1), S_j.transpose(1, 2)).squeeze(1)
        sim = sim.masked_fill(~valid_mask_batch, float("-inf"))

        w = torch.softmax(sim, dim=1)
        w = torch.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)

        weighted_sum = torch.bmm(w.unsqueeze(1), S_j).squeeze(1)

        has_valid = valid_mask_batch.any(dim=1)
        delta = torch.where(
            has_valid.unsqueeze(1),
            S_i - weighted_sum,
            torch.zeros_like(S_i)
        )

        grad_out[ids_batch] = delta

## 8. Run FUSE optimization

In [ ]:
lr = hyper_params["lr"]
lambda_sup = hyper_params["lambda_sup"]
lambda_semi = hyper_params["lambda_semi"]
T_epochs = hyper_params["T_epochs"]

print("\n--- Starting FUSE Optimization on Cora ---")
print("Hyperparameters:", hyper_params)
t0 = time.time()

with torch.no_grad():
    grad_total = torch.zeros_like(S)
    grad_mod = torch.zeros_like(S)
    grad_sup = torch.zeros_like(S)
    grad_semi = torch.zeros_like(S)

    for iteration in range(T_epochs):
        modularity_grad_prop(A_sparse, d, S, m, grad_mod)

        supervised_grad(S, labels_masked, is_labeled, num_classes, grad_sup)

        semi_sup_grad(S, walk_neigh, unlab_ids, grad_semi)

        grad_total.copy_(grad_mod)
        grad_total.add_(grad_sup, alpha=-lambda_sup)
        grad_total.add_(grad_semi, alpha=-lambda_semi)

        S.add_(grad_total, alpha=lr)

        Q, _ = torch.linalg.qr(S, mode="reduced")
        S.copy_(Q)
        del Q

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        mod_norm = torch.norm(grad_mod).item()
        sup_norm = torch.norm(grad_sup).item()
        semi_norm = torch.norm(grad_semi).item()
        total_grad_norm = torch.norm(grad_total).item()
        elapsed = time.time() - t0

        if (iteration + 1) % 10 == 0 or iteration == 0:
            print(
                f"Iter {iteration+1:3d}/{T_epochs} | "
                f"Mod: {mod_norm:9.6f} | "
                f"Sup: {sup_norm:8.4f} | "
                f"Semi: {semi_norm:8.4f} | "
                f"Total: {total_grad_norm:8.4f} | "
                f"Time: {elapsed:.1f}s"
            )

print(f"\nTotal structural generation time: {time.time() - t0:.4f}s")
print("FUSE optimization complete.")
print("Generated embedding matrix:", S.shape)


--- Starting FUSE Optimization on Cora ---
Hyperparameters: {'k': 145, 'lr': 0.31, 'lambda_sup': 0.6, 'lambda_semi': 1.9, 'T_epochs': 200, 'r': 20, 'L': 4, 'Lp': 1}
Iter   1/200 | Mod:  0.002797 | Sup:   4.2001 | Semi:   2.0237 | Total:   4.5960 | Time: 0.0s
Iter  10/200 | Mod:  0.002798 | Sup:   4.1982 | Semi:   2.0242 | Total:   4.5960 | Time: 0.1s
Iter  20/200 | Mod:  0.002799 | Sup:   4.1960 | Semi:   2.0246 | Total:   4.5961 | Time: 0.1s
Iter  30/200 | Mod:  0.002800 | Sup:   4.1938 | Semi:   2.0251 | Total:   4.5961 | Time: 0.1s
Iter  40/200 | Mod:  0.002801 | Sup:   4.1917 | Semi:   2.0256 | Total:   4.5962 | Time: 0.2s
Iter  50/200 | Mod:  0.002802 | Sup:   4.1895 | Semi:   2.0261 | Total:   4.5962 | Time: 0.2s
Iter  60/200 | Mod:  0.002803 | Sup:   4.1874 | Semi:   2.0266 | Total:   4.5963 | Time: 0.3s
Iter  70/200 | Mod:  0.002804 | Sup:   4.1852 | Semi:   2.0270 | Total:   4.5963 | Time: 0.3s
Iter  80/200 | Mod:  0.002805 | Sup:   4.1830 | Semi:   2.0275 | Total:   4.5964 |

In [ ]:
torch.save(r"S_cora", S)


AttributeError: expected 'f' to be string, path, or a file-like object with a 'write' attribute

In [ ]:
S_generated = S.detach()

train_idx = train_idx.to(device)
test_idx = test_idx.to(device)
labels_tensor = labels_tensor.to(device)

S_train = S_generated[train_idx]
y_train = labels_tensor[train_idx].long()

S_test = S_generated[test_idx]
y_test = labels_tensor[test_idx].long()

print(f"Train nodes: {S_train.shape[0]} | Test nodes: {S_test.shape[0]}")

# Standardize using train statistics only
mean = S_train.mean(dim=0, keepdim=True)
std = S_train.std(dim=0, keepdim=True) + 1e-7

S_train_scaled = (S_train - mean) / std
S_test_scaled = (S_test - mean) / std

Train nodes: 812 | Test nodes: 1896


### MLP

In [ ]:
batch_size = min(4096, len(train_idx))
train_dataset = TensorDataset(S_train_scaled, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


K = S_train.shape[1]
model = MLP(K, 256, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

epochs = 200

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / max(1, len(train_loader))

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Train Loss: {avg_train_loss:.4f}")

model.eval()
with torch.no_grad():
    test_logits = model(S_test_scaled)
    test_preds = test_logits.argmax(dim=1)
    test_acc = (test_preds == y_test).float().mean().item()
    test_f1_weighted = f1_score(
        y_test.detach().cpu().numpy(),
        test_preds.detach().cpu().numpy(),
        average="weighted"
    )

print(f"\nFinal Cora Test Accuracy using FUSE embeddings + MLP: {test_acc:.4f}")
print(f"Weighted Test F1 using FUSE embeddings + MLP:    {test_f1_weighted:.4f}")
print("\n--- MLP Test Classification Report ---")
print(classification_report(
    y_test.detach().cpu().numpy(),
    test_preds.detach().cpu().numpy(),
    digits=4
))

Epoch   1/200 | Train Loss: 2.0819
Epoch  10/200 | Train Loss: 0.2332
Epoch  20/200 | Train Loss: 0.0559
Epoch  30/200 | Train Loss: 0.0249
Epoch  40/200 | Train Loss: 0.0138
Epoch  50/200 | Train Loss: 0.0108
Epoch  60/200 | Train Loss: 0.0089
Epoch  70/200 | Train Loss: 0.0079
Epoch  80/200 | Train Loss: 0.0065
Epoch  90/200 | Train Loss: 0.0076
Epoch 100/200 | Train Loss: 0.0060
Epoch 110/200 | Train Loss: 0.0056
Epoch 120/200 | Train Loss: 0.0051
Epoch 130/200 | Train Loss: 0.0047
Epoch 140/200 | Train Loss: 0.0038
Epoch 150/200 | Train Loss: 0.0039
Epoch 160/200 | Train Loss: 0.0044
Epoch 170/200 | Train Loss: 0.0042
Epoch 180/200 | Train Loss: 0.0034
Epoch 190/200 | Train Loss: 0.0033
Epoch 200/200 | Train Loss: 0.0038

Final Cora Test Accuracy using FUSE embeddings + MLP: 0.7780
Weighted Test F1 using FUSE embeddings + MLP:    0.7797

--- MLP Test Classification Report ---
              precision    recall  f1-score   support

           0     0.6367    0.7195    0.6756       24

### 9.2 Multinomial Logistic Regression

In [ ]:

X_train_np = S_train_scaled.detach().cpu().numpy()
y_train_np = y_train.detach().cpu().numpy()

X_test_np = S_test_scaled.detach().cpu().numpy()
y_test_np = y_test.detach().cpu().numpy()

print("Training Logistic Regression...")
clf = LogisticRegression(
    max_iter=2000,
    multi_class="multinomial",
    solver="lbfgs",
    n_jobs=-1
)
clf.fit(X_train_np, y_train_np)

y_test_pred = clf.predict(X_test_np)

test_acc = accuracy_score(y_test_np, y_test_pred)
test_f1_weighted = f1_score(y_test_np, y_test_pred, average="weighted")

print(f"Test Accuracy:    {test_acc:.4f}")
print(f"Weighted Test F1: {test_f1_weighted:.4f}")

print("\n--- Logistic Regression Test Classification Report ---")
print(classification_report(y_test_np, y_test_pred, digits=4))

Training Logistic Regression...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Test Accuracy:    0.7780
Weighted Test F1: 0.7790

--- Logistic Regression Test Classification Report ---
              precision    recall  f1-score   support

           0     0.6374    0.7073    0.6705       246
           1     0.6726    0.7434    0.7063       152
           2     0.8614    0.8908    0.8758       293
           3     0.8170    0.7714    0.7935       573
           4     0.8315    0.7785    0.8042       298
           5     0.7778    0.7740    0.7759       208
           6     0.7360    0.7302    0.7331       126

    accuracy                         0.7780      1896
   macro avg     0.7620    0.7708    0.7656      1896
weighted avg     0.7816    0.7780    0.7790      1896

